In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# --- 1. LOAD DATA ---

try:
    df = pd.read_csv('air_traffic_data.csv')
    print("Loaded from air_traffic_data.csv")
except FileNotFoundError:
    np.random.seed(42)
    n = 100
    dom_flt = np.random.randint(500, 2000, n)
    int_flt = np.random.randint(100, 800, n)
    df = pd.DataFrame({
        'Dom_Pax': (dom_flt * np.random.uniform(80,  150, n) + np.random.normal(0, 1000, n)).astype(int),
        'Int_Pax': (int_flt * np.random.uniform(120, 250, n) + np.random.normal(0,  800, n)).astype(int),
        'Dom_Flt': dom_flt,
        'Int_Flt': int_flt,
        'Dom_RPM': (dom_flt * np.random.uniform(400, 800, n)).astype(int),
    })
    df['Pax'] = df['Dom_Pax'] + df['Int_Pax']
    df['Flt'] = df['Dom_Flt'] + df['Int_Flt']
    print("Generated sample data")

print(f"Shape: {df.shape}")
display(df.head())
print(df.describe())
print("Missing:", df.isnull().sum().sum())

# --- 2. CORRELATION HEATMAP ---

corr = df.corr()
print("\nCorrelation with Pax:")
print(corr['Pax'].sort_values(ascending=False))

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, square=True, linewidths=1)
plt.title('Air Traffic Correlation Matrix')
plt.tight_layout()
plt.show()

# --- 3. HYPOTHESIS TESTS ---

# Test 1: Are domestic and international passenger counts significantly different?
t_stat, p_dom_vs_int = stats.ttest_ind(df['Dom_Pax'], df['Int_Pax'])
print(f"\nT-test Dom_Pax vs Int_Pax:")
print(f"  t={t_stat:.4f}, p={p_dom_vs_int:.6f}")
print(f"  Mean Dom_Pax: {df['Dom_Pax'].mean():.0f} | Mean Int_Pax: {df['Int_Pax'].mean():.0f}")
print("  Significant difference" if p_dom_vs_int < 0.05 else "  No significant difference")

# Test 2: Is there a correlation between total flights and total passengers?
r_pax_flt, p_pax_flt = stats.pearsonr(df['Pax'], df['Flt'])
print(f"\nPearson correlation Pax vs Flt:")
print(f"  r={r_pax_flt:.4f}, p={p_pax_flt:.6f}")
print("  Significant correlation" if p_pax_flt < 0.05 else "  No significant correlation")

# --- 4. SIMPLE LINEAR REGRESSION: Flt → Pax ---

X_simple = df[['Flt']]
y = df['Pax']
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_simple, y, test_size=0.2, random_state=42)

simple_model = LinearRegression()
simple_model.fit(X_train_s, y_train_s)
y_pred_simple = simple_model.predict(X_test_s)

r2_simple   = r2_score(y_test_s, y_pred_simple)
rmse_simple = np.sqrt(mean_squared_error(y_test_s, y_pred_simple))
mae_simple  = mean_absolute_error(y_test_s, y_pred_simple)

print(f"\nSimple Regression (Flt → Pax):")
print(f"  R²={r2_simple:.4f} | RMSE={rmse_simple:.0f} | MAE={mae_simple:.0f}")
print(f"  Equation: Pax = {simple_model.intercept_:.0f} + {simple_model.coef_[0]:.2f} x Flt")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(X_test_s, y_test_s, alpha=0.6, label='Actual')
axes[0].plot(X_test_s, y_pred_simple, 'r-', label='Predicted')
axes[0].set_title('Simple Regression: Pax vs Flt')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

residuals_s = y_test_s - y_pred_simple
axes[1].scatter(y_pred_simple, residuals_s, alpha=0.6)
axes[1].axhline(0, color='r', linestyle='--', linewidth=2)
axes[1].set_title('Residuals — Simple Regression')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- 5. MULTIPLE LINEAR REGRESSION ---

features = ['Dom_Pax', 'Int_Pax', 'Dom_Flt', 'Int_Flt', 'Dom_RPM']
X_multi  = df[features]
y_multi  = df['Pax']

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_multi, y_multi, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_m)
X_test_scaled  = scaler.transform(X_test_m)

multi_model = LinearRegression()
multi_model.fit(X_train_scaled, y_train_m)
y_pred_multi = multi_model.predict(X_test_scaled)

r2_multi   = r2_score(y_test_m, y_pred_multi)
rmse_multi = np.sqrt(mean_squared_error(y_test_m, y_pred_multi))
mae_multi  = mean_absolute_error(y_test_m, y_pred_multi)

print(f"\nMultiple Regression:")
print(f"  R²={r2_multi:.4f} | RMSE={rmse_multi:.0f} | MAE={mae_multi:.0f}")

coef_df = pd.DataFrame({'Feature': features, 'Coefficient': multi_model.coef_})
print("\nFeature coefficients (standardized):")
print(coef_df.sort_values('Coefficient', key=abs, ascending=False).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test_m, y_pred_multi, alpha=0.6)
axes[0].plot([y_test_m.min(), y_test_m.max()], [y_test_m.min(), y_test_m.max()], 'r--', lw=2)
axes[0].set_title('Multiple Regression: Predicted vs Actual')
axes[0].grid(True, alpha=0.3)

residuals_m = y_test_m - y_pred_multi
axes[1].scatter(y_pred_multi, residuals_m, alpha=0.6)
axes[1].axhline(0, color='r', linestyle='--', linewidth=2)
axes[1].set_title('Residuals — Multiple Regression')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- 6. MODEL COMPARISON ---

print(f"\n{'Model':<22} {'R²':<10} {'RMSE':<12} {'MAE'}")
print(f"{'Simple Regression':<22} {r2_simple:<10.4f} {rmse_simple:<12.0f} {mae_simple:.0f}")
print(f"{'Multiple Regression':<22} {r2_multi:<10.4f} {rmse_multi:<12.0f} {mae_multi:.0f}")
print(f"\nR² improvement: {(r2_multi - r2_simple)*100:+.1f} percentage points")